# Discern · PS-1 Open-Set Person Re-ID under Low Inter-Class Variance (Market-1501)

**Run on Kaggle:** Add Data → `pengcw1/market-1501` · Accelerator → **GPU T4 ×2** (or P100) · Internet **ON** (ImageNet weights) → *Save Version → Save & Run All*.
Everything the frontend needs lands in `/kaggle/working/model/` (also zipped as `discern_model.zip`).

**Recipe:** ResNet-50 (last-stride 1) + GeM + BNNeck · label-smoothed CE + batch-hard triplet · **identity-level look-alike hard-negative mining** · GPU-resident data & augmentation (zero CPU bottleneck) · fp16 AMP · EMA · cosine LR auto-fitted to the time budget.
**Open-set head:** flip-TTA → multi-shot identity aggregation → margin-to-runner-up + query-adaptive z-score → Platt-calibrated confidence → operating-point thresholds (strict / balanced / lenient).
**Evaluation:** official Market-1501 CMC/mAP, open-set TAR-vs-FAR over repeated enrolled/unknown identity splits, look-alike (same-clothing proxy) subsets LV75/LV90, inference-side and training-side ablations.


In [1]:
# ============================================================================
# DISCERN — Open-Set Person Re-ID under low inter-class variance (Zed Digital PS-1)
# Cell 1 · Environment, configuration, hardware detection
# ============================================================================
import os, sys, glob, json, time, math, random, re, shutil, copy, zipfile, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from PIL import Image
warnings.filterwarnings("ignore")

SMOKE = os.environ.get("REID_SMOKE") == "1"            # developer self-test switch (leave off)
ON_KAGGLE = os.path.exists("/kaggle/working")
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("REID_OUT", "./work"))
MODEL_DIR = OUT / "model"
REPORT_DIR = MODEL_DIR / "reports"
for d in (MODEL_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    # ---- input / batch --------------------------------------------------------------
    H=288, W=144,                 # network input (Market crops are 128x64 -> upsampled on GPU)
    P_PER_GPU=16, K=4,            # identities x images per identity, per GPU (batch 64/GPU)
    MULTI_GPU=True,               # DataParallel when >1 GPU (Kaggle T4 x2)
    # ---- schedule -------------------------------------------------------------------
    MAX_EPOCHS=120, WARMUP=5,
    TRAIN_BUDGET_MIN=170,         # epochs auto-shrink if the GPU is slow (e.g. P100); never exceeds this
    LR=4e-4, WD=5e-4, EMA=0.999,
    # ---- losses ---------------------------------------------------------------------
    LABEL_SMOOTH=0.1, TRI_MARGIN=0.3, TRI_W=1.0,
    # ---- identity-level hard-negative mining (look-alike batches) -------------------
    MINE_START=10, MINE_EVERY=5, MINE_FRAC=0.75, MINE_TOPN=30,
    # ---- evaluation -----------------------------------------------------------------
    EVAL_EVERY=20, SEEDS=10,
    # ---- optional training-component ablations (adds ~3 short runs) ----------------
    RUN_TRAIN_ABLATIONS=True, ABL_EPOCHS=30,
    SEED=42,
)
if SMOKE:
    CFG.update(H=128, W=64, P_PER_GPU=4, K=4, MAX_EPOCHS=4, WARMUP=1, TRAIN_BUDGET_MIN=None,
               MINE_START=1, MINE_EVERY=1, EVAL_EVERY=2, SEEDS=3, ABL_EPOCHS=2, MINE_TOPN=8, RUN_TRAIN_ABLATIONS=os.environ.get("REID_ABL")=="1")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["SEED"])

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
if dev.type == "cuda":
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU{i}: {p.name} | {p.total_memory/2**30:.1f} GiB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif not SMOKE:
    print("!! No GPU found. Notebook settings -> Accelerator -> GPU T4 x2 (or P100), then restart.")
USE_AMP = dev.type == "cuda"
USE_DP = CFG["MULTI_GPU"] and N_GPU > 1
P = CFG["P_PER_GPU"] * (N_GPU if USE_DP else 1)
K = CFG["K"]
BATCH = P * K
H, W = CFG["H"], CFG["W"]
print(f"torch {torch.__version__} | AMP={USE_AMP} | DataParallel={USE_DP} | batch={BATCH} (P={P} x K={K}) | input={H}x{W}")
try:
    import psutil; print(f"CPU RAM: {psutil.virtual_memory().total/2**30:.1f} GiB total")
except Exception:
    pass


GPU0: Tesla T4 | 14.6 GiB
GPU1: Tesla T4 | 14.6 GiB
torch 2.11.0+cu128 | AMP=True | DataParallel=True | batch=128 (P=32 x K=4) | input=288x144
CPU RAM: 31.3 GiB total


In [2]:
# ============================================================================
# Cell 2 · Load Market-1501 fully into RAM (uint8, ~0.9 GB) — no DataLoader workers needed
# ============================================================================
def find_market_root():
    env = os.environ.get("REID_ROOT")
    if env: return env
    for base in ("/kaggle/input", "."):
        for dp, dn, fn in os.walk(base):
            if "bounding_box_train" in dn and "bounding_box_test" in dn and "query" in dn:
                return dp
            if dp.count(os.sep) - base.count(os.sep) > 5:
                dn[:] = []
    raise FileNotFoundError("Market-1501 not found. Add the Kaggle dataset 'pengcw1/market-1501' via Add Data.")

ROOT = find_market_root()
print("Market-1501 root:", ROOT)

def load_split(folder):
    files = sorted(glob.glob(os.path.join(ROOT, folder, "*.jpg")))
    pid = np.array([int(re.match(r"(-?\d+)_c(\d)", os.path.basename(f)).group(1)) for f in files])
    cam = np.array([int(re.match(r"(-?\d+)_c(\d)", os.path.basename(f)).group(2)) for f in files])
    def rd(f):
        im = Image.open(f).convert("RGB")
        if im.size != (64, 128): im = im.resize((64, 128), Image.BILINEAR)
        return np.asarray(im)
    with ThreadPoolExecutor(8) as ex:
        arr = np.stack(list(ex.map(rd, files)))
    return files, torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous(), pid, cam

t0 = time.time()
TR_FILES, TR_X, TR_PID, TR_CAM = load_split("bounding_box_train")
Q_FILES,  Q_X,  Q_PID,  Q_CAM  = load_split("query")
GA_FILES, GA_X, GA_PID, GA_CAM = load_split("bounding_box_test")
print(f"train {len(TR_X)} | query {len(Q_X)} | gallery(raw) {len(GA_X)} | loaded in {time.time()-t0:.0f}s")

# closed-set gallery: drop junk (-1). distractors (pid 0) stay, as in the official protocol
keep = GA_PID != -1
GC_X, GC_PID, GC_CAM = GA_X[keep], GA_PID[keep], GA_CAM[keep]
# open-set gallery: real people only (pid > 0)
keep_os = GA_PID > 0
G_FILES = [f for f, k in zip(GA_FILES, keep_os) if k]
G_X, G_PID, G_CAM = GA_X[keep_os], GA_PID[keep_os], GA_CAM[keep_os]

# train labels -> 0..C-1
TR_IDS = np.unique(TR_PID); N_CLS = len(TR_IDS)
TR_Y = np.searchsorted(TR_IDS, TR_PID)
ITERS = max(20, len(TR_X) // BATCH) if not SMOKE else 6
print(f"train identities: {N_CLS} | iters/epoch: {ITERS} | open-set ids: {len(np.unique(Q_PID))} | open-set gallery imgs: {len(G_X)}")

# the whole training set lives on the GPU as uint8 (~320 MB) -> augmentation is done on-GPU
TRX_GPU = TR_X.to(dev)
TRY_GPU = torch.as_tensor(TR_Y, device=dev)


Market-1501 root: /kaggle/input/datasets/pengcw1/market-1501/Market-1501-v15.09.15
train 12936 | query 3368 | gallery(raw) 19732 | loaded in 27s
train identities: 751 | iters/epoch: 101 | open-set ids: 750 | open-set gallery imgs: 13115


In [3]:
# ============================================================================
# Cell 3 · Model (ResNet-50 last-stride-1 + GeM + BNNeck), GPU augmentation,
#          batch-hard triplet, identity-level hard-negative sampler, EMA
# ============================================================================
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
PRETRAINED = True

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1) * p); self.eps = eps
    def forward(self, x):
        return x.float().clamp(min=self.eps).pow(self.p).mean((2, 3)).pow(1.0 / self.p)

class ReIDNet(nn.Module):
    """Single compact backbone -> 2048-d embedding. Train returns (feat, bn_feat, logits); eval returns bn_feat."""
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        w = None
        if pretrained:
            try:
                w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1
                r = torchvision.models.resnet50(weights=w)
            except Exception as e:
                print("!! ImageNet weights unavailable (turn Internet ON in Kaggle settings). Training from scratch.", repr(e)[:120])
                r = torchvision.models.resnet50(weights=None)
        else:
            r = torchvision.models.resnet50(weights=None)
        r.layer4[0].conv2.stride = (1, 1)            # last stride 1 -> 2x finer feature map (big re-ID win)
        r.layer4[0].downsample[0].stride = (1, 1)
        self.backbone = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3, r.layer4)
        self.pool = GeM()
        self.bn = nn.BatchNorm1d(2048); self.bn.bias.requires_grad_(False)     # BNNeck
        self.cls = nn.Linear(2048, num_classes, bias=False); nn.init.normal_(self.cls.weight, std=0.001)
    def forward(self, x):
        f = self.pool(self.backbone(x)); b = self.bn(f)
        return (f, b, self.cls(b)) if self.training else b

# ---------------- GPU augmentation (one fused affine + colour + erasing pass) ----------------
def _u(n, lo, hi):
    return torch.empty(n, device=dev).uniform_(lo, hi)

def gpu_augment(xu8):
    B = xu8.shape[0]
    x = xu8.float() / 255.0
    s = _u(B, 0.88, 1.12); asp = _u(B, 0.94, 1.06)
    sx, sy = s * asp, s / asp
    fl = (torch.rand(B, device=dev) < 0.5).float() * 2 - 1
    rot = _u(B, -5, 5) * math.pi / 180
    tx, ty = _u(B, -0.10, 0.10), _u(B, -0.06, 0.06)
    th = torch.zeros(B, 2, 3, device=dev)
    th[:, 0, 0] = sx * torch.cos(rot) * fl; th[:, 0, 1] = -sx * torch.sin(rot); th[:, 0, 2] = tx
    th[:, 1, 0] = sy * torch.sin(rot) * fl; th[:, 1, 1] = sy * torch.cos(rot);  th[:, 1, 2] = ty
    grid = F.affine_grid(th, (B, 3, H, W), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    # colour: brightness / contrast / saturation (no hue shift - colour is identity information)
    x = x * _u(B, 0.8, 1.2).view(B, 1, 1, 1)
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * _u(B, 0.8, 1.2).view(B, 1, 1, 1) + m
    g = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]); x = g + (x - g) * _u(B, 0.8, 1.2).view(B, 1, 1, 1)
    x = ((x.clamp(0, 1) - MEAN) / STD)
    # random erasing (p=0.5)
    area = H * W * _u(B, 0.02, 0.25); ar = torch.exp(_u(B, math.log(0.3), math.log(3.3)))
    eh = (area * ar).sqrt().round().clamp(4, H - 2); ew = (area / ar).sqrt().round().clamp(4, W - 2)
    y0 = (torch.rand(B, device=dev) * (H - eh)).floor(); x0 = (torch.rand(B, device=dev) * (W - ew)).floor()
    ys = torch.arange(H, device=dev).view(1, H, 1); xs = torch.arange(W, device=dev).view(1, 1, W)
    msk = (ys >= y0.view(-1, 1, 1)) & (ys < (y0 + eh).view(-1, 1, 1)) & (xs >= x0.view(-1, 1, 1)) & (xs < (x0 + ew).view(-1, 1, 1))
    msk = msk & (torch.rand(B, device=dev) < 0.5).view(-1, 1, 1)
    x = x.masked_fill(msk.unsqueeze(1), 0.0)
    return x.contiguous(memory_format=torch.channels_last)

def prep_eval(xu8, flip=False):
    x = F.interpolate(xu8.float() / 255.0, size=(H, W), mode="bilinear", align_corners=False)
    if flip: x = x.flip(3)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)

# ---------------- losses ----------------
def batch_hard_triplet(f, y, margin):
    f = F.normalize(f.float(), dim=1)
    d = (2 - 2 * f @ f.T).clamp(min=1e-12).sqrt()
    same = y[:, None] == y[None, :]
    d_ap = d.masked_fill(~same, -1.0).max(1).values          # hardest positive
    d_an = d.masked_fill(same, 1e9).min(1).values            # hardest negative
    return F.relu(d_ap - d_an + margin).mean()

# ---------------- PK sampler with identity-level hard-negative mining ----------------
class PKSampler:
    """Random PxK batches; once centroids are available, MINE_FRAC of batches are built from
    clusters of mutually look-alike identities, so batch-hard triplets see genuinely hard negatives."""
    def __init__(self, labels, P, K, seed):
        self.n = int(labels.max()) + 1
        self.idx = [np.where(labels == c)[0] for c in range(self.n)]
        self.P, self.K = P, K; self.rng = np.random.RandomState(seed); self.neigh = None
    def set_centroids(self, cent):
        sim = cent @ cent.T; sim.fill_diagonal_(-2)
        self.neigh = sim.topk(min(CFG["MINE_TOPN"], self.n - 1), dim=1).indices.cpu().numpy()
    def batch(self, mining=True):
        P, rng = self.P, self.rng
        if mining and self.neigh is not None and rng.rand() < CFG["MINE_FRAC"]:
            na = 2 if P >= 8 else 1; per = P // na; chosen = []
            for a in rng.choice(self.n, na, replace=False):
                nb = rng.choice(self.neigh[a], min(per - 1, self.neigh.shape[1]), replace=False)
                chosen += [int(a), *map(int, nb)]
            chosen = list(dict.fromkeys(chosen))
            while len(chosen) < P:
                c = int(rng.randint(self.n))
                if c not in chosen: chosen.append(c)
            pids = np.array(chosen[:P])
        else:
            pids = rng.choice(self.n, P, replace=False)
        idx = np.concatenate([rng.choice(self.idx[c], self.K, replace=len(self.idx[c]) < self.K) for c in pids])
        return idx

class EMA:
    def __init__(self, net, decay):
        self.m = copy.deepcopy(net).eval(); self.decay = decay
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, net, step):
        d = min(self.decay, (1 + step) / (10 + step))
        torch._foreach_lerp_(list(self.m.parameters()), [p.detach() for p in net.parameters()], 1 - d)
        torch._foreach_lerp_([b for b in self.m.buffers() if b.dtype.is_floating_point],
                             [b for b in net.buffers() if b.dtype.is_floating_point], 1 - d)

def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)


In [4]:
# ============================================================================
# Cell 4 · Feature extraction + official closed-set Market-1501 metrics (CMC / mAP)
# ============================================================================
@torch.no_grad()
def extract(model, X, bs=256, tta=False):
    """returns (plain, tta) L2-normalised embeddings on GPU. tta = mean of image & horizontal flip."""
    model.eval(); P_, T_ = [], []
    for i in range(0, len(X), bs):
        xb = X[i:i + bs].to(dev, non_blocking=True)
        with autocast(): f = model(prep_eval(xb)).float()
        f = F.normalize(f, dim=1); P_.append(f)
        if tta:
            with autocast(): f2 = model(prep_eval(xb, True)).float()
            T_.append(F.normalize(f + F.normalize(f2, dim=1), dim=1))
    return torch.cat(P_), (torch.cat(T_) if tta else None)

@torch.no_grad()
def closed_set_metrics(fq, fg):
    sim = fq @ fg.T
    gp = torch.as_tensor(GC_PID, device=dev); gc = torch.as_tensor(GC_CAM, device=dev)
    qp = torch.as_tensor(Q_PID, device=dev); qc = torch.as_tensor(Q_CAM, device=dev)
    aps, firsts = [], []
    for i in range(len(fq)):
        valid = ~((gp == qp[i]) & (gc == qc[i]))
        lab = (gp[valid] == qp[i])
        order = sim[i][valid].argsort(descending=True)
        pos = lab[order].nonzero().squeeze(1).float()
        if pos.numel() == 0: continue
        aps.append(((torch.arange(len(pos), device=dev) + 1) / (pos + 1)).mean()); firsts.append(pos[0])
    f = torch.stack(firsts)
    return dict(mAP=float(torch.stack(aps).mean()) * 100, R1=float((f < 1).float().mean()) * 100,
                R5=float((f < 5).float().mean()) * 100, R10=float((f < 10).float().mean()) * 100)

def quick_eval(model, tta=False):
    fq = extract(model, Q_X, tta=tta); fg = extract(model, GC_X, tta=tta)
    k = 1 if tta else 0
    return closed_set_metrics(fq[k], fg[k])


In [5]:
# ============================================================================
# Cell 5 · Training engine (AMP fp16, channels_last, GPU-resident data, EMA, resume, time-budget aware)
# ============================================================================
def lr_at(e, total):
    base, wu = CFG["LR"], CFG["WARMUP"]
    if e < wu: return base * (0.02 + 0.98 * e / wu)
    pr = min(1.0, (e - wu) / max(1, total - wu))
    return 1e-6 + 0.5 * (base - 1e-6) * (1 + math.cos(math.pi * pr))

@torch.no_grad()
def refresh_neighbors(net, sampler):
    f, _ = extract(net, TR_X, bs=256)
    cent = torch.zeros(N_CLS, f.shape[1], device=dev).index_add_(0, TRY_GPU, f)
    sampler.set_centroids(F.normalize(cent, dim=1))

def train(tag, epochs_max, use_tri=True, mining=True, use_ema=True, budget_min=None, resume=False):
    set_seed(CFG["SEED"])
    net = ReIDNet(N_CLS, pretrained=PRETRAINED).to(dev).to(memory_format=torch.channels_last)
    model = nn.DataParallel(net) if USE_DP else net
    opt = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], lr=CFG["LR"], weight_decay=CFG["WD"])
    try: scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    ema = EMA(net, CFG["EMA"]) if use_ema else None
    sampler = PKSampler(TR_Y, P, K, CFG["SEED"])
    total, ep, step, hist, ep_times = epochs_max, 0, 0, [], []
    ck = OUT / f"ckpt_{tag}.pt"
    if resume and ck.exists():
        s = torch.load(ck, map_location=dev)
        net.load_state_dict(s["net"]); opt.load_state_dict(s["opt"]); scaler.load_state_dict(s["scaler"])
        if ema: ema.m.load_state_dict(s["ema"])
        ep, total, step, hist = s["ep"], s["total"], s["step"], s["hist"]
        print(f"[{tag}] resumed at epoch {ep}/{total}")
    t_start = time.time()
    while ep < total:
        t_ep = time.time()
        if mining and ep >= CFG["MINE_START"] and (ep - CFG["MINE_START"]) % CFG["MINE_EVERY"] == 0:
            refresh_neighbors(ema.m if ema else net, sampler)
        model.train(); net.train()
        li = torch.zeros((), device=dev); lt = torch.zeros((), device=dev)
        for it in range(ITERS):
            lr = lr_at(ep + it / ITERS, total)
            for g in opt.param_groups: g["lr"] = lr
            idx = torch.as_tensor(sampler.batch(mining), device=dev)
            x = gpu_augment(TRX_GPU[idx]); y = TRY_GPU[idx]
            with autocast():
                f, b, logits = model(x)
            l_id = F.cross_entropy(logits.float(), y, label_smoothing=CFG["LABEL_SMOOTH"])
            l_tr = batch_hard_triplet(f, y, CFG["TRI_MARGIN"]) if use_tri else torch.zeros((), device=dev)
            loss = l_id + CFG["TRI_W"] * l_tr
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if ema: ema.update(net, step)
            step += 1; li += l_id.detach(); lt += l_tr.detach()
        ep += 1; dt = time.time() - t_ep; ep_times.append(dt)
        rec = dict(epoch=ep, lr=lr, id_loss=float(li) / ITERS, tri_loss=float(lt) / ITERS, sec=dt)
        if budget_min and ep == 3:                      # fit the schedule to the time budget (cosine adapts)
            est = int(budget_min * 60 / (np.mean(ep_times[1:3]) * 1.12))
            total = int(min(epochs_max, max(min(30, epochs_max), est)))
            print(f"[{tag}] {np.mean(ep_times[1:3]):.1f}s/epoch -> training for {total} epochs")
        if ep % CFG["EVAL_EVERY"] == 0 or ep == total:
            m = quick_eval(ema.m if ema else net); rec.update({f"cs_{k}": v for k, v in m.items()})
            print(f"[{tag}] ep {ep}/{total}  closed-set  mAP {m['mAP']:.2f}  R1 {m['R1']:.2f}")
        hist.append(rec)
        print(f"[{tag}] ep {ep:3d}/{total}  lr {lr:.2e}  id {rec['id_loss']:.3f}  tri {rec['tri_loss']:.3f}  "
              f"{dt:5.1f}s  {ITERS*BATCH/dt:6.0f} img/s")
        if ep % 10 == 0 or ep == total:
            torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                            ema=ema.m.state_dict() if ema else None, ep=ep, total=total, step=step, hist=hist), ck)
    print(f"[{tag}] done in {(time.time()-t_start)/60:.1f} min")
    return dict(net=net, ema=ema.m if ema else None, hist=hist)


In [6]:
# ============================================================================
# Cell 6 · MAIN TRAINING RUN  (full recipe: CE+label-smoothing, batch-hard triplet,
#          identity-level hard-negative mining, GeM + BNNeck, EMA, cosine LR)
# ============================================================================
main = train("main", CFG["MAX_EPOCHS"], use_tri=True, mining=True, use_ema=True,
             budget_min=CFG["TRAIN_BUDGET_MIN"], resume=True)

cands = {"ema": main["ema"], "raw": main["net"]}
scores = {k: quick_eval(m, tta=True) for k, m in cands.items()}
for k, v in scores.items(): print(f"final {k:3s}: mAP {v['mAP']:.2f}  R1 {v['R1']:.2f}  R5 {v['R5']:.2f}  (flip-TTA)")
best_key = max(scores, key=lambda k: scores[k]["mAP"])
FINAL = cands[best_key].eval(); CLOSED_FINAL = scores[best_key]
print("selected:", best_key, CLOSED_FINAL)

# features for all later analysis (plain + flip-TTA)
fq_p, fq_t = extract(FINAL, Q_X,  tta=True)
fg_p, fg_t = extract(FINAL, G_X,  tta=True)           # real-person gallery (pid>0)
FEATS = {"plain": (fq_p, fg_p), "tta": (fq_t, fg_t)}
torch.save(FINAL.state_dict(), MODEL_DIR / "discern_reid_r50.pt")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s] 


[main] ep   1/120  lr 8.56e-05  id 6.488  tri 0.348   99.4s     130 img/s
[main] ep   2/120  lr 1.64e-04  id 5.179  tri 0.286   96.3s     134 img/s
[main] 96.2s/epoch -> training for 94 epochs
[main] ep   3/94  lr 2.42e-04  id 3.501  tri 0.238   96.0s     135 img/s
[main] ep   4/94  lr 3.21e-04  id 2.376  tri 0.221   96.3s     134 img/s
[main] ep   5/94  lr 3.99e-04  id 1.849  tri 0.217   95.4s     135 img/s
[main] ep   6/94  lr 4.00e-04  id 1.643  tri 0.215   95.9s     135 img/s
[main] ep   7/94  lr 4.00e-04  id 1.494  tri 0.209   95.4s     135 img/s
[main] ep   8/94  lr 3.99e-04  id 1.410  tri 0.196   95.5s     135 img/s
[main] ep   9/94  lr 3.98e-04  id 1.346  tri 0.180   95.5s     135 img/s
[main] ep  10/94  lr 3.97e-04  id 1.337  tri 0.173   95.5s     135 img/s
[main] ep  11/94  lr 3.96e-04  id 1.560  tri 0.231  114.5s     113 img/s
[main] ep  12/94  lr 3.94e-04  id 1.487  tri 0.247   95.4s     135 img/s
[main] ep  13/94  lr 3.92e-04  id 1.367  tri 0.229   95.6s     135 img/s
[mai

In [7]:
# ============================================================================
# Cell 7 · OPEN-SET protocol, look-alike (low inter-class variance) subsets, scoring rules
# ----------------------------------------------------------------------------
# Protocol (repeated over SEEDS random identity splits of the 750 test identities):
#   enrolled  = 50% of identities (gallery = all their gallery images)
#   unknown   = the other 50% (never enrolled; every probe of theirs MUST be rejected)
#   genuine probe  : query of an enrolled id  -> correct accept needs top-1 id correct AND score > threshold
#   unknown probe  : query of an unenrolled id -> any score > threshold is a FALSE ACCEPT
#   TAR = correct accepts / genuine probes ; FAR = false accepts / unknown probes
# Calibration (Platt) is fitted on a held-out half of probes (unknown ids disjoint), metrics use the rest.
# Look-alike subsets are defined independently of the network from colour statistics of upper/lower body
# (a proxy for "same uniform / similar clothes") -> LV75 / LV90 = probes whose nearest enrolled
# look-alike lies in the top 25% / 10% of colour similarity.
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

IDS_ALL = np.unique(Q_PID); NQ = len(Q_PID)
QP_T, QC_T = torch.as_tensor(Q_PID, device=dev), torch.as_tensor(Q_CAM, device=dev)

# ---------- colour-based look-alike structure (network independent) ----------
def rgb2hsv(x):
    r, g, b = x[:, 0], x[:, 1], x[:, 2]
    mx = x.max(1).values; mn = x.min(1).values; d = mx - mn; m = d > 1e-6
    rc = (mx == r) & m; gc_ = (mx == g) & m & ~rc; bc = m & ~rc & ~gc_
    h = torch.zeros_like(mx)
    h = torch.where(rc, ((g - b) / (d + 1e-6)) % 6, h)
    h = torch.where(gc_, (b - r) / (d + 1e-6) + 2, h)
    h = torch.where(bc, (r - g) / (d + 1e-6) + 4, h)
    s = torch.where(mx > 1e-6, d / (mx + 1e-6), torch.zeros_like(mx))
    return h / 6.0, s, mx

@torch.no_grad()
def color_hist(X, bs=1024):
    out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(dev).float() / 255.0
        h, s, v = rgb2hsv(x)
        idx = (h * 8).long().clamp(0, 7) * 16 + (s * 4).long().clamp(0, 3) * 4 + (v * 4).long().clamp(0, 3)
        regs = []
        for a, b in ((20, 64), (64, 122)):                      # upper body / lower body bands (of 128 rows)
            ii = idx[:, a:b].flatten(1)
            hh = torch.zeros(len(ii), 128, device=dev).scatter_add_(1, ii, torch.ones_like(ii, dtype=torch.float))
            regs.append(hh / hh.sum(1, keepdim=True))
        out.append(torch.stack(regs, 1))
    return torch.cat(out)

_hq, _hg = color_hist(Q_X), color_hist(G_X)
_hid = torch.zeros(len(IDS_ALL), 2, 128, device=dev)
_cnt = torch.zeros(len(IDS_ALL), device=dev)
for hh, pp in ((_hq, Q_PID), (_hg, G_PID)):
    ii = torch.as_tensor(np.searchsorted(IDS_ALL, pp), device=dev)
    _hid.index_add_(0, ii, hh); _cnt.index_add_(0, ii, torch.ones(len(ii), device=dev))
_hid = _hid / _cnt.view(-1, 1, 1)
COLOR_SIM = (torch.einsum("irb,jrb->ij", _hid.sqrt(), _hid.sqrt()) / 2).cpu().numpy()   # Bhattacharyya
np.fill_diagonal(COLOR_SIM, -1)
QID_IDX = np.searchsorted(IDS_ALL, Q_PID)

# ---------- identity-aggregated scores for every probe ----------
def _aggregate(S3, agg):
    if agg == "max":
        return S3.max(-1).values
    k = int(agg[3:]); v = S3.topk(min(k, S3.shape[-1]), dim=-1).values; ok = v > -1.5
    r = (v * ok).sum(-1) / ok.sum(-1).clamp(min=1)
    return torch.where(ok.any(-1), r, torch.full_like(r, -1.0))

@torch.no_grad()
def probe_stats(feat_key, agg, enrolled):
    Fq, Fg = FEATS[feat_key]
    cols = np.where(np.isin(G_PID, enrolled))[0]
    order = cols[np.argsort(G_PID[cols], kind="stable")]; gp = G_PID[order]; gc = G_CAM[order]
    ids, counts = np.unique(gp, return_counts=True); E, M = len(ids), counts.max()
    im = np.full((E, M), -1); st = 0
    for r, c in enumerate(counts): im[r, :c] = np.arange(st, st + c); st += c
    sim = Fq @ Fg[order].T
    sim = sim.masked_fill((QP_T[:, None] == torch.as_tensor(gp, device=dev)[None]) &
                          (QC_T[:, None] == torch.as_tensor(gc, device=dev)[None]), -2.0)   # official same-id/same-cam exclusion
    simp = torch.cat([sim, torch.full((NQ, 1), -2.0, device=dev)], 1)
    im = torch.as_tensor(im, device=dev); im = torch.where(im < 0, simp.shape[1] - 1, im)
    S = torch.cat([_aggregate(simp[i:i + 256][:, im], agg) for i in range(0, NQ, 256)])
    top = S.topk(2, dim=1)
    s1, s2 = top.values[:, 0], top.values[:, 1]
    z = (s1 - S.mean(1)) / (S.std(1) + 1e-6)
    return dict(s1=s1.cpu().numpy(), m=(s1 - s2).cpu().numpy(), z=z.cpu().numpy(), pred=ids[top.indices[:, 0].cpu().numpy()])

def make_split(seed):
    rng = np.random.RandomState(seed); perm = rng.permutation(IDS_ALL); h = len(perm) // 2
    E, U = perm[:h], perm[h:]; U_cal = U[:len(U) // 2]
    genuine = np.isin(Q_PID, E)
    is_cal = np.zeros(NQ, bool); is_cal[genuine] = rng.rand(genuine.sum()) < 0.5; is_cal[np.isin(Q_PID, U_cal)] = True
    lv_id = np.where(np.isin(IDS_ALL, E)[None, :], COLOR_SIM, -1).max(1)           # best colour-twin among enrolled (self excluded)
    return dict(E=E, genuine=genuine, is_cal=is_cal, lv=lv_id[QID_IDX], seed=seed)

# ---------- operating-point metrics ----------
def op_metrics(score, genuine, correct, mask):
    g = genuine & mask; u = (~genuine) & mask; ng, nu = g.sum(), u.sum()
    cs = np.sort(score[g & correct])[::-1]; us = np.sort(score[u])[::-1]
    r = dict(n_gen=int(ng), n_unk=int(nu))
    if ng == 0 or nu == 0 or len(cs) == 0:
        return {**r, **{k: float("nan") for k in ("FAR@TAR80", "FAR@TAR90", "TAR@FAR0.1%", "TAR@FAR1%", "TAR@FAR10%", "AUROC", "rank1_open")}}
    for T in (0.80, 0.90):
        k = int(math.ceil(T * ng))
        r[f"FAR@TAR{int(T*100)}"] = float((us >= cs[k - 1]).mean()) if k <= len(cs) and nu else 1.0
    for F_ in (0.001, 0.01, 0.1):
        j = int(math.floor(F_ * nu)); t = us[min(j, len(us) - 1)]
        r[f"TAR@FAR{F_*100:g}%"] = float((cs > t).sum() / ng) if ng else 0.0
    pos, neg = score[g & correct], score[u]
    r["AUROC"] = float(roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg])) if len(pos) and len(neg) else float("nan")
    r["rank1_open"] = float((g & correct).sum() / max(1, ng))
    return r

def curve(score, genuine, correct, mask, n=400):
    g = genuine & mask; u = (~genuine) & mask
    cs, us = np.sort(score[g & correct]), np.sort(score[u])
    th = np.quantile(np.r_[cs, us], np.linspace(0, 1, n))
    return 1 - np.searchsorted(us, th, side="right") / len(us), 1 - np.searchsorted(cs, th, side="right") / g.sum()   # FAR, TAR

def fit_lr(st, split, mask, feats):
    X = np.stack([st[f] for f in feats], 1); y = (split["genuine"] & (st["pred"] == Q_PID)).astype(int)
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X[mask], y[mask])

def lr_scores(lr, st, feats):
    X = np.stack([st[f] for f in feats], 1)
    return lr.decision_function(X), lr.predict_proba(X)[:, 1]

SUBSETS = {"full": None, "LV75": 75, "LV90": 90}
def eval_subsets(score, split, st, which="eval"):
    genuine, correct = split["genuine"], (st["pred"] == Q_PID)
    base = ~split["is_cal"] if which == "eval" else split["is_cal"]
    out = {}
    for name, pc in SUBSETS.items():
        mk = base if pc is None else base & (split["lv"] >= np.percentile(split["lv"][base], pc))
        out[name] = op_metrics(score, genuine, correct, mk)
    return out


In [8]:
# ============================================================================
# Cell 8 · Open-set analysis: ROC / FAR-vs-TAR, look-alike subsets, inference-side ablations, calibration
# ============================================================================
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SEEDS = list(range(CFG["SEEDS"])); SPLITS = [make_split(s) for s in SEEDS]
COMBOS = [("plain", "max"), ("tta", "max"), ("tta", "top2"), ("tta", "top3")]
STATS = [{c: probe_stats(c[0], c[1], sp["E"]) for c in COMBOS} for sp in SPLITS]

# choose the multi-shot aggregation on the CALIBRATION half only (no peeking at eval probes)
def cal_far90(agg):
    v = []
    for sp, stt in zip(SPLITS, STATS):
        st = stt[("tta", agg)]; v.append(eval_subsets(st["s1"], sp, st, "cal")["full"]["FAR@TAR90"])
    return float(np.nanmean(v))
AGG_SCORES = {a: cal_far90(a) for a in ("max", "top2", "top3")}
BEST_AGG = min(AGG_SCORES, key=AGG_SCORES.get)
print("aggregation selection on calibration probes (FAR@TAR90, lower=better):", {k: round(v, 4) for k, v in AGG_SCORES.items()}, "->", BEST_AGG)

ABL_ROWS = [  # (label, feat, agg, scoring)
    ("A0 baseline: single pass, max-sim, global threshold", "plain", "max", "s1"),
    ("A1 + flip test-time augmentation",                     "tta",   "max", "s1"),
    (f"A2 + multi-shot identity aggregation ({BEST_AGG})",   "tta", BEST_AGG, "s1"),
    ("A3 + margin to runner-up identity  [s1, margin]",      "tta", BEST_AGG, "lr_sm"),
    ("A4 + query-adaptive z-score (full Discern rule)",      "tta", BEST_AGG, "lr_smz"),
]
FEATS_SETS = {"lr_sm": ["s1", "m"], "lr_smz": ["s1", "m", "z"]}
RAW = {}   # pooled scores per row for curves
TABLE = []
for label, fk, ag, sc in ABL_ROWS:
    per_seed = []; pooled = {k: [] for k in ("score", "genuine", "correct", "mask_full", "mask_lv90")}
    for sp, stt in zip(SPLITS, STATS):
        st = stt[(fk, ag)]
        score = st["s1"] if sc == "s1" else lr_scores(fit_lr(st, sp, sp["is_cal"], FEATS_SETS[sc]), st, FEATS_SETS[sc])[0]
        per_seed.append(eval_subsets(score, sp, st))
        ev = ~sp["is_cal"]; lv90 = ev & (sp["lv"] >= np.percentile(sp["lv"][ev], 90))
        for k, v in (("score", score), ("genuine", sp["genuine"]), ("correct", st["pred"] == Q_PID), ("mask_full", ev), ("mask_lv90", lv90)):
            pooled[k].append(v[ev])
    RAW[label] = {k: np.concatenate(v) for k, v in pooled.items()}
    row = {"config": label}
    for sub in SUBSETS:
        for met in ("FAR@TAR90", "FAR@TAR80", "TAR@FAR1%", "TAR@FAR0.1%", "AUROC"):
            vals = np.array([r[sub][met] for r in per_seed]); row[f"{sub}|{met}"] = (float(np.nanmean(vals)), float(np.nanstd(vals)))
    TABLE.append(row)

def fmt(t): return f"{t[0]*100:.1f}±{t[1]*100:.1f}"
md = ["| Config | FAR@TAR90 full | FAR@TAR90 LV75 | FAR@TAR90 LV90 | TAR@FAR1% full | TAR@FAR1% LV90 | AUROC full | AUROC LV90 |", "|---|---|---|---|---|---|---|---|"]
for r in TABLE:
    md.append(f"| {r['config']} | {fmt(r['full|FAR@TAR90'])} | {fmt(r['LV75|FAR@TAR90'])} | {fmt(r['LV90|FAR@TAR90'])} | "
              f"{fmt(r['full|TAR@FAR1%'])} | {fmt(r['LV90|TAR@FAR1%'])} | {r['full|AUROC'][0]:.4f} | {r['LV90|AUROC'][0]:.4f} |")
INFER_ABL_MD = "\n".join(md); print(INFER_ABL_MD)
(REPORT_DIR / "ablation_inference.md").write_text(INFER_ABL_MD)

# ---------- figure: ROC (TAR vs FAR) + score distributions ----------
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
cols = ["#9aa0a6", "#c58af9", "#5b8def", "#12a594", "#e5484d"]
for (label, *_), c in zip(ABL_ROWS, cols):
    R = RAW[label]
    for j, (mk, ls) in enumerate((("mask_full", "-"), ("mask_lv90", "--"))):
        far, tar = curve(R["score"], R["genuine"], R["correct"], R[mk])
        ax[0].plot(far, tar, ls, color=c, lw=1.8, label=f"{label.split(' ')[0]} {'full' if j == 0 else 'LV90'}")
ax[0].set_xscale("log"); ax[0].set_xlim(1e-4, 1); ax[0].set_ylim(0, 1)
ax[0].set_xlabel("False-accept rate (unenrolled probes)"); ax[0].set_ylabel("True-accept rate (correct ID)")
ax[0].set_title("Open-set ROC  (solid: all   dashed: look-alike LV90)"); ax[0].grid(alpha=.3); ax[0].legend(fontsize=7, ncol=2)
R = RAW[ABL_ROWS[-1][0]]; g = R["genuine"] & R["correct"]; u = ~R["genuine"]
bins = np.linspace(np.percentile(R["score"], 0.5), np.percentile(R["score"], 99.5), 60)
ax[1].hist(R["score"][g], bins, alpha=.6, color="#12a594", label="genuine (correct ID)", density=True)
ax[1].hist(R["score"][u], bins, alpha=.6, color="#e5484d", label="unenrolled", density=True)
ax[1].hist(R["score"][u & R["mask_lv90"]], bins, alpha=.6, color="#f5a623", label="unenrolled · LV90 look-alikes", density=True)
ax[1].set_title("Full-rule score distributions"); ax[1].legend(fontsize=8); ax[1].set_xlabel("decision score")
rows = [r for r in TABLE]
xs = np.arange(len(rows)); wd = .38
ax[2].bar(xs - wd/2, [r["full|FAR@TAR90"][0] * 100 for r in rows], wd, color="#5b8def", label="all probes")
ax[2].bar(xs + wd/2, [r["LV90|FAR@TAR90"][0] * 100 for r in rows], wd, color="#f5a623", label="look-alike LV90")
ax[2].set_xticks(xs); ax[2].set_xticklabels([r["config"].split(" ")[0] for r in rows]); ax[2].set_ylabel("FAR (%) at TAR = 90%")
ax[2].set_title("Ablation: false-accept at fixed recall (lower is better)"); ax[2].legend(); ax[2].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(REPORT_DIR / "open_set_roc_ablation.png", dpi=150); plt.show()

# ---------- final deployable calibrators: pooled fit over all seeds, 3 fallbacks ----------
POOL = {k: [] for k in ("s1", "m", "z", "y", "genuine", "lv90")}
for sp, stt in zip(SPLITS, STATS):
    st = stt[("tta", BEST_AGG)]
    for k in ("s1", "m", "z"): POOL[k].append(st[k])
    POOL["y"].append((sp["genuine"] & (st["pred"] == Q_PID)).astype(int)); POOL["genuine"].append(sp["genuine"])
    POOL["lv90"].append(sp["lv"] >= np.percentile(sp["lv"], 90))
POOL = {k: np.concatenate(v) for k, v in POOL.items()}
CALIBRATORS = {}
for name, feats in (("s1", ["s1"]), ("s1_margin", ["s1", "m"]), ("s1_margin_z", ["s1", "m", "z"])):
    X = np.stack([POOL[f] for f in feats], 1)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X, POOL["y"])
    sc, lrm = pipe.named_steps["standardscaler"], pipe.named_steps["logisticregression"]
    p = pipe.predict_proba(X)[:, 1]; unk = ~POOL["genuine"].astype(bool); us = np.sort(p[unk])[::-1]
    ops = {}
    for nm, far in (("strict", 0.001), ("balanced", 0.01), ("lenient", 0.05)):
        t = float(us[min(int(far * len(us)), len(us) - 1)])
        ops[nm] = dict(threshold=t, target_far=far, far=float((p[unk] > t).mean()),
                       far_lookalike_lv90=float((p[unk & POOL["lv90"]] > t).mean()), tar=float(((p > t) & (POOL["y"] == 1)).sum() / POOL["genuine"].sum()))
    CALIBRATORS[name] = dict(features=feats, mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coef=lrm.coef_[0].tolist(),
                             intercept=float(lrm.intercept_[0]), operating_points=ops)
    print(name, {k: (round(v["threshold"], 3), f"FAR {v['far']*100:.2f}%", f"TAR {v['tar']*100:.1f}%", f"LV90-FAR {v['far_lookalike_lv90']*100:.2f}%") for k, v in ops.items()})

# reliability diagram of the full rule on held-out probes (cal-fit -> eval)
P_, Y_ = [], []
for sp, stt in zip(SPLITS, STATS):
    st = stt[("tta", BEST_AGG)]; lr = fit_lr(st, sp, sp["is_cal"], ["s1", "m", "z"]); ev = ~sp["is_cal"]
    P_.append(lr_scores(lr, st, ["s1", "m", "z"])[1][ev]); Y_.append((sp["genuine"] & (st["pred"] == Q_PID)).astype(int)[ev])
P_, Y_ = np.concatenate(P_), np.concatenate(Y_)
bn = np.linspace(0, 1, 11); bi = np.clip(np.digitize(P_, bn) - 1, 0, 9)
ece = sum(abs(P_[bi == b].mean() - Y_[bi == b].mean()) * (bi == b).mean() for b in range(10) if (bi == b).any())
BRIER = float(np.mean((P_ - Y_) ** 2)); print(f"calibration on held-out probes: ECE {ece:.4f} | Brier {BRIER:.4f}")
plt.figure(figsize=(4.6, 4.4)); plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.plot([P_[bi == b].mean() for b in range(10) if (bi == b).any()], [Y_[bi == b].mean() for b in range(10) if (bi == b).any()], "o-", color="#5b8def")
plt.xlabel("predicted confidence"); plt.ylabel("empirical accuracy of accepts"); plt.title(f"Reliability (ECE {ece:.3f})"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(REPORT_DIR / "calibration_reliability.png", dpi=150); plt.show()

OPEN_SUMMARY = {r["config"]: {k: v for k, v in r.items() if k != "config"} for r in TABLE}


aggregation selection on calibration probes (FAR@TAR90, lower=better): {'max': 0.1513, 'top2': 0.1548, 'top3': 0.1636} -> max
| Config | FAR@TAR90 full | FAR@TAR90 LV75 | FAR@TAR90 LV90 | TAR@FAR1% full | TAR@FAR1% LV90 | AUROC full | AUROC LV90 |
|---|---|---|---|---|---|---|---|
| A0 baseline: single pass, max-sim, global threshold | 14.5±3.3 | 23.0±9.2 | 32.1±26.8 | 54.1±14.7 | 44.0±20.5 | 0.9628 | 0.9422 |
| A1 + flip test-time augmentation | 13.0±3.8 | 19.4±6.8 | 29.7±26.8 | 54.9±14.9 | 46.4±22.4 | 0.9649 | 0.9446 |
| A2 + multi-shot identity aggregation (max) | 13.0±3.8 | 19.4±6.8 | 29.7±26.8 | 54.9±14.9 | 46.4±22.4 | 0.9649 | 0.9446 |
| A3 + margin to runner-up identity  [s1, margin] | 10.8±3.2 | 18.8±8.1 | 29.9±25.6 | 58.0±15.1 | 44.3±24.6 | 0.9697 | 0.9409 |
| A4 + query-adaptive z-score (full Discern rule) | 10.8±3.2 | 18.4±8.5 | 30.1±26.0 | 57.7±14.5 | 43.0±23.6 | 0.9697 | 0.9402 |
s1 {'strict': (0.989, 'FAR 0.09%', 'TAR 14.2%', 'LV90-FAR 0.65%'), 'balanced': (0.942, 'FAR 0.

In [9]:
# ============================================================================
# Cell 9 · TRAINING-component ablations (short runs, same eval protocol).  Set RUN_TRAIN_ABLATIONS=False to skip.
# ============================================================================
def open_set_quick(model, seeds=3):
    global FEATS
    saved = FEATS
    fq = extract(model, Q_X, tta=True); fg = extract(model, G_X, tta=True)
    FEATS = {"plain": (fq[0], fg[0]), "tta": (fq[1], fg[1])}
    try:
        res = {"full": [], "LV90": []}; tar = []
        for sd in range(seeds):
            sp = make_split(sd); st = probe_stats("tta", BEST_AGG, sp["E"])
            lr = fit_lr(st, sp, sp["is_cal"], ["s1", "m", "z"]); sc = lr_scores(lr, st, ["s1", "m", "z"])[0]
            r = eval_subsets(sc, sp, st)
            for k in res: res[k].append((r[k]["FAR@TAR90"], r[k]["TAR@FAR1%"]))
        cs = closed_set_metrics(fq[1], extract(model, GC_X, tta=True)[1])
        return dict(mAP=cs["mAP"], R1=cs["R1"], far90_full=np.nanmean([a for a, _ in res["full"]]) * 100, far90_lv90=np.nanmean([a for a, _ in res["LV90"]]) * 100,
                    tar1_lv90=np.nanmean([b for _, b in res["LV90"]]) * 100)
    finally:
        FEATS = saved

TRAIN_ABL = {}
if CFG["RUN_TRAIN_ABLATIONS"]:
    variants = [("T0 ID loss only (CE+LS)",            dict(use_tri=False, mining=False, use_ema=False)),
                ("T1 + batch-hard triplet",            dict(use_tri=True,  mining=False, use_ema=False)),
                ("T2 + identity-level look-alike mining", dict(use_tri=True, mining=True, use_ema=False)),
                ("T3 + EMA weights (full recipe)",     dict(use_tri=True,  mining=True, use_ema=True))]
    for name, kw in variants:
        r = train("abl_" + name.split(" ")[0], CFG["ABL_EPOCHS"], budget_min=None, resume=False, **kw)
        TRAIN_ABL[name] = open_set_quick(r["ema"] if r["ema"] is not None else r["net"])
        print(name, {k: round(v, 2) for k, v in TRAIN_ABL[name].items()})
        del r; torch.cuda.empty_cache()
    md = ["| Training config (%d epochs) | mAP | R1 | FAR@TAR90 all | FAR@TAR90 LV90 | TAR@FAR1% LV90 |" % CFG["ABL_EPOCHS"], "|---|---|---|---|---|---|"]
    for n, v in TRAIN_ABL.items():
        md.append(f"| {n} | {v['mAP']:.1f} | {v['R1']:.1f} | {v['far90_full']:.2f} | {v['far90_lv90']:.2f} | {v['tar1_lv90']:.1f} |")
    TRAIN_ABL_MD = "\n".join(md); print(TRAIN_ABL_MD); (REPORT_DIR / "ablation_training.md").write_text(TRAIN_ABL_MD)
else:
    TRAIN_ABL_MD = "(training ablations skipped)"


[abl_T0] ep   1/30  lr 8.56e-05  id 6.498  tri 0.000   95.4s     136 img/s
[abl_T0] ep   2/30  lr 1.64e-04  id 5.255  tri 0.000   95.3s     136 img/s
[abl_T0] ep   3/30  lr 2.42e-04  id 3.607  tri 0.000   95.2s     136 img/s
[abl_T0] ep   4/30  lr 3.21e-04  id 2.441  tri 0.000   95.1s     136 img/s
[abl_T0] ep   5/30  lr 3.99e-04  id 1.871  tri 0.000   94.9s     136 img/s
[abl_T0] ep   6/30  lr 3.98e-04  id 1.649  tri 0.000   95.2s     136 img/s
[abl_T0] ep   7/30  lr 3.94e-04  id 1.477  tri 0.000   95.1s     136 img/s
[abl_T0] ep   8/30  lr 3.86e-04  id 1.383  tri 0.000   94.9s     136 img/s
[abl_T0] ep   9/30  lr 3.75e-04  id 1.320  tri 0.000   95.0s     136 img/s
[abl_T0] ep  10/30  lr 3.62e-04  id 1.300  tri 0.000   95.1s     136 img/s
[abl_T0] ep  11/30  lr 3.46e-04  id 1.257  tri 0.000   95.4s     135 img/s
[abl_T0] ep  12/30  lr 3.28e-04  id 1.240  tri 0.000   95.7s     135 img/s
[abl_T0] ep  13/30  lr 3.08e-04  id 1.219  tri 0.000   95.5s     135 img/s
[abl_T0] ep  14/30  lr 2.

TypeError: not enough arguments for format string

In [10]:
md = [f"| Training config ({CFG['ABL_EPOCHS']} epochs) | mAP | R1 | FAR@TAR90 all | FAR@TAR90 LV90 | TAR@FAR1% LV90 |", "|---|---|---|---|---|---|"]
for n, v in TRAIN_ABL.items():
    md.append(f"| {n} | {v['mAP']:.1f} | {v['R1']:.1f} | {v['far90_full']:.2f} | {v['far90_lv90']:.2f} | {v['tar1_lv90']:.1f} |")
TRAIN_ABL_MD = "\n".join(md); print(TRAIN_ABL_MD)
(REPORT_DIR / "ablation_training.md").write_text(TRAIN_ABL_MD)

| Training config (30 epochs) | mAP | R1 | FAR@TAR90 all | FAR@TAR90 LV90 | TAR@FAR1% LV90 |
|---|---|---|---|---|---|
| T0 ID loss only (CE+LS) | 83.5 | 93.5 | 14.92 | 49.05 | 47.4 |
| T1 + batch-hard triplet | 85.1 | 93.5 | 17.23 | 53.96 | 39.1 |
| T2 + identity-level look-alike mining | 85.0 | 94.1 | 13.64 | 37.82 | 38.3 |
| T3 + EMA weights (full recipe) | 85.1 | 93.9 | 14.59 | 31.66 | 38.0 |


399

In [11]:
# ============================================================================
# Cell 10 · EXPORT the model folder that the frontend/backend links to
# ============================================================================
# ---- 10.1 embedder: ONNX (+TorchScript). Input = RGB float32 [0,1], NCHW, already resized to HxW. Output = L2-normalised 2048-d.
class ExportNet(nn.Module):
    def __init__(self, net):
        super().__init__(); self.net = net
        self.register_buffer("mean", MEAN.detach().cpu().clone()); self.register_buffer("std", STD.detach().cpu().clone())
    def forward(self, x):
        return F.normalize(self.net((x - self.mean) / self.std), dim=1)

exp = ExportNet(copy.deepcopy(FINAL).cpu().float().eval()).eval()
dummy = torch.rand(2, 3, H, W)
onnx_path = MODEL_DIR / "discern_embedder.onnx"; ONNX_OK = False
try:
    try: import onnx, onnxruntime as ort
    except Exception:
        os.system(f"{sys.executable} -m pip install -q onnx onnxruntime"); import onnx, onnxruntime as ort
    kw = dict(input_names=["images"], output_names=["embedding"], opset_version=17,
              dynamic_axes={"images": {0: "batch"}, "embedding": {0: "batch"}})
    try: torch.onnx.export(exp, dummy, str(onnx_path), dynamo=False, **kw)
    except TypeError: torch.onnx.export(exp, dummy, str(onnx_path), **kw)
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    a = sess.run(None, {"images": dummy.numpy()})[0]
    with torch.no_grad(): b = exp(dummy).numpy()
    print(f"ONNX exported ({onnx_path.stat().st_size/2**20:.0f} MB) | max |onnx - torch| = {np.abs(a-b).max():.2e}")
    ONNX_OK = np.abs(a - b).max() < 1e-3
except Exception as e:
    print("!! ONNX export failed:", repr(e)[:300], "-> TorchScript/PyTorch weights still exported")
with torch.no_grad():
    torch.jit.trace(exp, dummy).save(str(MODEL_DIR / "discern_embedder.torchscript.pt"))
torch.save(dict(state_dict=FINAL.state_dict(), num_classes=N_CLS, input_hw=[H, W], arch="resnet50-laststride1-gem-bnneck"),
           MODEL_DIR / "discern_reid_r50.pt")

# ---- 10.2 pooled arrays of every real-person image (query + gallery) with TTA embeddings
ALL_F = torch.cat([FEATS["tta"][0], FEATS["tta"][1]]); ALL_PID = np.r_[Q_PID, G_PID]; ALL_CAM = np.r_[Q_CAM, G_CAM]
ALL_FILE = list(Q_FILES) + list(G_FILES); ALL_ROLE = np.r_[np.zeros(len(Q_PID)), np.ones(len(G_PID))]
cal1 = CALIBRATORS["s1"]
def p_from_s1(s):
    z = (s - cal1["mean"][0]) / cal1["scale"][0]; return float(1 / (1 + math.exp(-(cal1["coef"][0] * z + cal1["intercept"]))))

# ---- 10.3 Look-alike Explorer data: most confusable different-identity pairs (+ typical genuine pair for contrast)
SAMPLES = MODEL_DIR / "samples"; shutil.rmtree(SAMPLES, ignore_errors=True); SAMPLES.mkdir()
ids_u = np.unique(ALL_PID); rows = np.searchsorted(ids_u, ALL_PID)
cent = torch.zeros(len(ids_u), ALL_F.shape[1], device=dev).index_add_(0, torch.as_tensor(rows, device=dev), ALL_F)
cent = F.normalize(cent, dim=1); csim = (cent @ cent.T).cpu().numpy(); np.fill_diagonal(csim, -1)
iu = np.triu_indices(len(ids_u), 1); order = np.argsort(-csim[iu]); used = {}; PAIRS = []
for o in order:
    a, b = ids_u[iu[0][o]], ids_u[iu[1][o]]
    if used.get(a, 0) >= 2 or used.get(b, 0) >= 2: continue
    used[a] = used.get(a, 0) + 1; used[b] = used.get(b, 0) + 1
    A, B = np.where(ALL_PID == a)[0], np.where(ALL_PID == b)[0]
    sm = (ALL_F[A] @ ALL_F[B].T).cpu().numpy(); i, j = np.unravel_index(sm.argmax(), sm.shape)
    # typical genuine pair of identity a (cross-camera, median similarity)
    ga = (ALL_F[A] @ ALL_F[A].T).cpu().numpy(); cm = ALL_CAM[A][:, None] != ALL_CAM[A][None, :]
    iuA = np.triu_indices(len(A), 1); okp = cm[iuA]
    if okp.sum() == 0: continue
    vals = ga[iuA][okp]; k = np.argsort(vals)[len(vals) // 2]; gi, gj = iuA[0][okp][k], iuA[1][okp][k]
    files = {"a": ALL_FILE[A[i]], "b": ALL_FILE[B[j]], "ga1": ALL_FILE[A[gi]], "ga2": ALL_FILE[A[gj]]}
    for key, f in files.items(): shutil.copy(f, SAMPLES / os.path.basename(f))
    PAIRS.append(dict(rank=len(PAIRS) + 1, id_a=int(a), id_b=int(b),
                      image_a=f"samples/{os.path.basename(files['a'])}", image_b=f"samples/{os.path.basename(files['b'])}",
                      impostor_similarity=float(sm[i, j]), impostor_confidence=p_from_s1(float(sm[i, j])),
                      identity_centroid_similarity=float(csim[iu[0][o], iu[1][o]]),
                      colour_similarity=float(COLOR_SIM[iu[0][o], iu[1][o]]),
                      genuine_image_1=f"samples/{os.path.basename(files['ga1'])}", genuine_image_2=f"samples/{os.path.basename(files['ga2'])}",
                      genuine_similarity=float(vals[k]), genuine_confidence=p_from_s1(float(vals[k]))))
    if len(PAIRS) >= 60: break
(MODEL_DIR / "lookalike_explorer.json").write_text(json.dumps(dict(
    description="Most confusable different-identity pairs in the Market-1501 test identities, ranked by identity-centroid cosine similarity. "
                "impostor_* = the closest cross-identity image pair; genuine_* = a typical cross-camera pair of identity A. Confidence = calibrated s1-only Platt score.",
    operating_points=cal1["operating_points"], pairs=PAIRS), indent=1))
fig, axs = plt.subplots(3, 4, figsize=(12, 9.5))
for ax, pr in zip(axs.ravel(), PAIRS[:12]):
    ia, ib = Image.open(MODEL_DIR / pr["image_a"]), Image.open(MODEL_DIR / pr["image_b"])
    canvas = Image.new("RGB", (ia.width + ib.width + 6, ia.height), "white"); canvas.paste(ia, (0, 0)); canvas.paste(ib, (ia.width + 6, 0))
    ax.imshow(canvas); ax.axis("off"); ax.set_title(f"IDs {pr['id_a']} vs {pr['id_b']}  cos={pr['impostor_similarity']:.2f}", fontsize=9)
plt.suptitle("Closest different-person pairs (the false-accept risk)", y=1.0); plt.tight_layout()
plt.savefig(REPORT_DIR / "lookalike_pairs.png", dpi=130); plt.show()

# ---- 10.4 demo gallery + probes so the UI works instantly
DEMO = MODEL_DIR / "demo"; shutil.rmtree(DEMO, ignore_errors=True); (DEMO / "images").mkdir(parents=True)
rng = np.random.RandomState(7); n_demo = min(40, len(ids_u) // 4)
enr = rng.choice(ids_u, n_demo, replace=False); enr_rows = np.searchsorted(ids_u, enr)
cs_ = csim.copy(); cs_[:, np.searchsorted(ids_u, enr)] = -9                      # candidates for unknown = not enrolled
unk = []
for r_ in enr_rows:
    for cand in np.argsort(-cs_[r_]):
        if ids_u[cand] not in unk and ids_u[cand] not in enr: unk.append(int(ids_u[cand])); break
man = {"gallery": [], "probes": []}; gal_emb = []; prb_emb = []
for pid in enr:
    cand = np.where((ALL_PID == pid) & (ALL_ROLE == 1))[0]; seen = set(); pick = []
    for c in cand:
        if ALL_CAM[c] not in seen: seen.add(ALL_CAM[c]); pick.append(c)
        if len(pick) == 3: break
    for c in pick:
        shutil.copy(ALL_FILE[c], DEMO / "images" / os.path.basename(ALL_FILE[c]))
        man["gallery"].append(dict(identity=f"ID-{int(pid):04d}", file="images/" + os.path.basename(ALL_FILE[c]), pid=int(pid))); gal_emb.append(ALL_F[c].cpu().numpy())
    qs = np.where((ALL_PID == pid) & (ALL_ROLE == 0))[0]
    if len(qs):
        c = qs[0]; shutil.copy(ALL_FILE[c], DEMO / "images" / os.path.basename(ALL_FILE[c]))
        man["probes"].append(dict(file="images/" + os.path.basename(ALL_FILE[c]), pid=int(pid), truth="enrolled", expected=f"ID-{int(pid):04d}")); prb_emb.append(ALL_F[c].cpu().numpy())
for pid in unk:
    qs = np.where((ALL_PID == pid) & (ALL_ROLE == 0))[0]
    if len(qs):
        c = qs[0]; shutil.copy(ALL_FILE[c], DEMO / "images" / os.path.basename(ALL_FILE[c]))
        man["probes"].append(dict(file="images/" + os.path.basename(ALL_FILE[c]), pid=int(pid), truth="unenrolled_lookalike", expected=None)); prb_emb.append(ALL_F[c].cpu().numpy())
np.savez_compressed(DEMO / "demo_embeddings.npz", gallery=np.array(gal_emb, np.float32), probes=np.array(prb_emb, np.float32))
(DEMO / "demo_manifest.json").write_text(json.dumps(man, indent=1))

# ---- 10.5 decision config (the single source of truth for the backend)
cfg_out = dict(
    model=dict(name="Discern-R50-BNNeck", architecture="ResNet-50 (last stride 1) + GeM + BNNeck", embedding_dim=2048,
               onnx="discern_embedder.onnx", torchscript="discern_embedder.torchscript.pt", pytorch_weights="discern_reid_r50.pt", onnx_validated=bool(ONNX_OK)),
    preprocess=dict(input_size_hw=[H, W], colour="RGB", range="float32 0..1, NCHW", resize="stretch the person crop to WxH (bilinear); no padding",
                    normalisation="done INSIDE the exported model (ImageNet mean/std) - do not normalise again"),
    embedding=dict(l2_normalised=True, similarity="cosine (dot product)", flip_tta=True,
                   tta_recipe="e = normalize(embed(x) + embed(horizontal_flip(x)))  <- used for enrolment AND probes"),
    matching=dict(aggregation=BEST_AGG, aggregation_note="identity score = mean of the top-k image similarities of that identity (max when k=1)",
                  decision="confidence = sigmoid(coef . standardise([s1, margin, z]) + intercept); accept iff confidence > operating-point threshold; else REJECT as 'unknown'",
                  features=dict(s1="best identity score", margin="s1 minus runner-up identity score", z="(s1 - mean(all identity scores)) / std(all identity scores)"),
                  min_identities_full_calibrator=10, fallback="s1_margin if >=2 identities, s1 if 1 identity"),
    calibrators=CALIBRATORS,
    default_operating_point="balanced",
    protocol=dict(dataset="Market-1501 (test identities, 750)", split=f"50% enrolled / 50% unknown, {len(SEEDS)} random splits",
                  lookalike_subsets="LV75/LV90 = probes whose nearest enrolled colour-twin is in the top 25% / 10% of upper+lower-body HSV similarity",
                  note="Thresholds were fitted with ~375 enrolled identities; gallery size shifts score statistics - keep the UI slider."),
)
(MODEL_DIR / "decision_config.json").write_text(json.dumps(cfg_out, indent=1))

# ---- 10.6 reports
hist = main["hist"]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot([h["epoch"] for h in hist], [h["id_loss"] for h in hist], label="ID (CE+LS)"); ax[0].plot([h["epoch"] for h in hist], [h["tri_loss"] for h in hist], label="triplet"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot([h["epoch"] for h in hist], [h["lr"] for h in hist]); ax[1].set_title("Learning rate")
ev_ = [h for h in hist if "cs_mAP" in h]; ax[2].plot([h["epoch"] for h in ev_], [h["cs_mAP"] for h in ev_], "o-", label="mAP"); ax[2].plot([h["epoch"] for h in ev_], [h["cs_R1"] for h in ev_], "s-", label="Rank-1"); ax[2].legend(); ax[2].set_title("Market-1501 closed-set")
plt.tight_layout(); plt.savefig(REPORT_DIR / "training_curves.png", dpi=140); plt.show()
with open(REPORT_DIR / "training_log.json", "w") as f: json.dump(hist, f)
report = dict(closed_set_market1501=CLOSED_FINAL, selected_weights=best_key, best_aggregation=BEST_AGG,
              calibration_heldout=dict(ECE=float(ece), Brier=BRIER), open_set_ablation=OPEN_SUMMARY, training_ablation=TRAIN_ABL,
              operating_points={k: v["operating_points"] for k, v in CALIBRATORS.items()},
              hardware=dict(gpus=[torch.cuda.get_device_name(i) for i in range(N_GPU)], epochs=len(hist), batch=BATCH, input_hw=[H, W]))
(REPORT_DIR / "eval_report.json").write_text(json.dumps(report, indent=1))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 79.8 MB/s eta 0:00:00
ONNX exported (90 MB) | max |onnx - torch| = 8.94e-08


9174

In [12]:
# ============================================================================
# Cell 11 · Backend reference implementation + integration README + zip
# ============================================================================
REF = '''"""discern_inference.py - reference backend for the Discern open-set re-ID model (copy into your backend).
pip install onnxruntime-gpu (or onnxruntime) numpy pillow
"""
import json, os
import numpy as np
from PIL import Image
import onnxruntime as ort


class Discern:
    def __init__(self, model_dir=".", providers=None):
        self.cfg = json.load(open(os.path.join(model_dir, "decision_config.json")))
        avail = ort.get_available_providers()
        prov = providers or [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in avail]
        self.sess = ort.InferenceSession(os.path.join(model_dir, self.cfg["model"]["onnx"]), providers=prov)
        self.H, self.W = self.cfg["preprocess"]["input_size_hw"]
        self.rows = {}          # identity -> list of row indices into self.emb
        self.emb = np.zeros((0, self.cfg["model"]["embedding_dim"]), np.float32)

    # ---------- embedding ----------
    def _prep(self, img):
        a = np.asarray(img.convert("RGB").resize((self.W, self.H), Image.BILINEAR), dtype=np.float32) / 255.0
        return a.transpose(2, 0, 1)

    def embed(self, imgs):
        """imgs: list of PIL person crops -> (N, 2048) L2-normalised, flip-TTA."""
        x = np.stack([self._prep(i) for i in imgs])
        e1 = self.sess.run(None, {"images": x})[0]
        e2 = self.sess.run(None, {"images": np.ascontiguousarray(x[..., ::-1])})[0]
        e = e1 + e2
        return (e / np.linalg.norm(e, axis=1, keepdims=True)).astype(np.float32)

    # ---------- gallery ----------
    def add_embeddings(self, identity, emb):
        start = len(self.emb)
        self.emb = np.concatenate([self.emb, emb.astype(np.float32)])
        self.rows.setdefault(identity, []).extend(range(start, start + len(emb)))

    def enroll(self, identity, imgs):
        self.add_embeddings(identity, self.embed(imgs))

    def remove(self, identity):          # rebuild after delete
        keep = {k: v for k, v in self.rows.items() if k != identity}
        old = self.emb; self.emb = np.zeros((0, old.shape[1]), np.float32); self.rows = {}
        for k, v in keep.items(): self.add_embeddings(k, old[v])

    # ---------- open-set decision ----------
    def identify(self, img_or_emb, op=None):
        q = self.embed([img_or_emb])[0] if isinstance(img_or_emb, Image.Image) else img_or_emb
        names = list(self.rows)
        if not names:
            return dict(decision="reject", reason="empty gallery", identity=None, confidence=0.0)
        agg = self.cfg["matching"]["aggregation"]; k = 1 if agg == "max" else int(agg[3:])
        sims = self.emb @ q
        S = np.array([np.sort(sims[self.rows[n]])[::-1][:k].mean() for n in names])
        order = np.argsort(-S); s1 = float(S[order[0]]); n = len(names)
        s2 = float(S[order[1]]) if n > 1 else 0.0
        m = s1 - s2; z = (s1 - S.mean()) / (S.std() + 1e-6)
        cal_name = "s1_margin_z" if n >= self.cfg["matching"]["min_identities_full_calibrator"] else ("s1_margin" if n >= 2 else "s1")
        cal = self.cfg["calibrators"][cal_name]
        vals = {"s1": s1, "m": m, "z": z}
        x = (np.array([vals[f] for f in cal["features"]]) - np.array(cal["mean"])) / np.array(cal["scale"])
        conf = float(1 / (1 + np.exp(-(np.dot(cal["coef"], x) + cal["intercept"]))))
        op = op or self.cfg["default_operating_point"]; thr = cal["operating_points"][op]["threshold"]
        top = [dict(identity=names[i], score=float(S[i])) for i in order[:5]]
        accept = conf > thr
        return dict(decision="accept" if accept else "reject", identity=names[order[0]] if accept else None,
                    best_candidate=names[order[0]], confidence=conf, threshold=thr, calibrator=cal_name,
                    s1=s1, margin=m, z=float(z), operating_point=op, top5=top)


if __name__ == "__main__":      # replay the bundled demo
    import sys
    d = sys.argv[1] if len(sys.argv) > 1 else "."
    D = Discern(d); man = json.load(open(os.path.join(d, "demo", "demo_manifest.json")))
    for g in man["gallery"]:
        D.enroll(g["identity"], [Image.open(os.path.join(d, "demo", g["file"]))])
    ok = tot = fa = unk = 0
    for p in man["probes"]:
        r = D.identify(Image.open(os.path.join(d, "demo", p["file"])))
        if p["truth"] == "enrolled": tot += 1; ok += (r["identity"] == p["expected"])
        else: unk += 1; fa += (r["decision"] == "accept")
    print(f"known probes correctly accepted: {ok}/{tot} | look-alike strangers falsely accepted: {fa}/{unk}")
'''
(MODEL_DIR / "discern_inference.py").write_text(REF)

full = CALIBRATORS["s1_margin_z"]["operating_points"]
README = f"""# Discern model bundle — integration contract

Open-set person re-identification (Zed Digital PS-1). Everything the app needs is in this folder.

| file | purpose |
|---|---|
| `discern_embedder.onnx` | **the model.** input `images` float32 `[N,3,{H},{W}]` RGB in 0..1 -> output `embedding` `[N,2048]` L2-normalised (ImageNet normalisation is inside) |
| `decision_config.json` | preprocessing, TTA rule, aggregation, calibrators and operating-point thresholds — **single source of truth** |
| `discern_inference.py` | reference backend (`Discern.embed / enroll / identify`), onnxruntime only |
| `lookalike_explorer.json` + `samples/` | 60 most confusable different-person pairs (+ a genuine pair for contrast) for the Look-alike Explorer page |
| `demo/` | pre-computed demo gallery/probes (`demo_manifest.json`, `demo_embeddings.npz`, images) so the UI works with zero setup |
| `reports/` | `eval_report.json`, `ablation_inference.md`, `ablation_training.md`, ROC / calibration / training PNGs |
| `discern_reid_r50.pt`, `discern_embedder.torchscript.pt` | PyTorch weights / TorchScript (optional) |

## Pipeline (what the backend must do)
1. Person crop (RGB) -> resize/stretch to **{W}x{H}** (bilinear) -> float 0..1 -> NCHW.
2. `e = normalize(onnx(x) + onnx(flip_horizontal(x)))`  (flip-TTA, used for enrolment **and** probes).
3. Gallery = list of `(identity, embedding)`; identity score = mean of top-k cosine sims (`aggregation` in config: **{BEST_AGG}**).
4. Features: `s1` (best identity), `margin = s1 - runner_up`, `z = (s1 - mean(S)) / std(S)`.
5. `confidence = sigmoid(coef · ((f-mean)/scale) + intercept)` using calibrator `s1_margin_z` (>=10 identities), else `s1_margin`, else `s1`.
6. Accept iff `confidence > threshold[operating_point]`, otherwise answer **UNKNOWN** (never force a match).

## Operating points (calibrator `s1_margin_z`, pooled Market-1501 open-set protocol)
| name | threshold | false-accept | look-alike (LV90) false-accept | true-accept |
|---|---|---|---|---|
""" + "\n".join(f"| {k} | {v['threshold']:.3f} | {v['far']*100:.2f}% | {v['far_lookalike_lv90']*100:.2f}% | {v['tar']*100:.1f}% |" for k, v in full.items()) + f"""

Closed-set Market-1501: mAP **{CLOSED_FINAL['mAP']:.1f}**, Rank-1 **{CLOSED_FINAL['R1']:.1f}** (flip-TTA, single query, no re-ranking).
Calibration on held-out probes: ECE **{ece:.3f}**.

## Suggested REST surface
`POST /api/enroll` (identity, images[]) · `POST /api/identify` (image, op?) · `DELETE /api/identity/{{id}}` · `GET /api/config` (decision_config.json)
· `GET /api/lookalikes` (lookalike_explorer.json) · `GET /api/metrics` (reports/eval_report.json) · `POST /api/demo/load`.
Note: thresholds were fitted with ~375 enrolled identities; keep an operating-point selector (strict / balanced / lenient) in the UI.
"""
(MODEL_DIR / "README_INTEGRATION.md").write_text(README)

# ---- sanity test of the shipped reference backend against the notebook's own embeddings
if ONNX_OK:
    try:
        sys.path.insert(0, str(MODEL_DIR)); import importlib; di = importlib.import_module("discern_inference")
        D = di.Discern(str(MODEL_DIR), providers=["CPUExecutionProvider"])
        mf = json.load(open(DEMO / "demo_manifest.json")); e = D.embed([Image.open(DEMO / mf["gallery"][0]["file"])])[0]
        ref = np.load(DEMO / "demo_embeddings.npz")["gallery"][0]
        print(f"backend-vs-notebook embedding cosine: {float(e @ ref):.4f}  (PIL vs torch resize; >0.97 expected)")
        os.system(f"{sys.executable} {MODEL_DIR}/discern_inference.py {MODEL_DIR}")
    except Exception as ex:
        print("reference backend self-test skipped:", repr(ex)[:200])

for f in OUT.glob("ckpt_*.pt"): f.unlink()
shutil.make_archive(str(OUT / "discern_model"), "zip", MODEL_DIR)
print("\nMODEL FOLDER:", MODEL_DIR)
for p in sorted(MODEL_DIR.rglob("*")):
    if p.is_file() and "samples/" not in str(p) and "demo/images" not in str(p): print(f"  {str(p.relative_to(MODEL_DIR)):45s} {p.stat().st_size/2**20:8.2f} MB")
print("zip:", OUT / "discern_model.zip")


backend-vs-notebook embedding cosine: 0.9998  (PIL vs torch resize; >0.97 expected)
known probes correctly accepted: 40/40 | look-alike strangers falsely accepted: 2/40

MODEL FOLDER: /kaggle/working/model
  README_INTEGRATION.md                             0.00 MB
  __pycache__/discern_inference.cpython-313.pyc     0.01 MB
  decision_config.json                              0.00 MB
  demo/demo_embeddings.npz                          1.44 MB
  demo/demo_manifest.json                           0.02 MB
  discern_embedder.onnx                            89.64 MB
  discern_embedder.torchscript.pt                  96.16 MB
  discern_inference.py                              0.00 MB
  discern_reid_r50.pt                              95.88 MB
  lookalike_explorer.json                           0.03 MB
  reports/ablation_inference.md                     0.00 MB
  reports/ablation_training.md                      0.00 MB
  reports/calibration_reliability.png               0.05 MB
  reports/eval